In [4]:
import pandas as pd
import requests
import json

In [7]:
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.10813/dados"
params = {
    "formato": "json",
    "dataInicial": "01/01/2023",
    "dataFinal": "31/12/2024"
}

response = requests.get(url, params=params)
cambio = pd.DataFrame(response.json())
cambio["data"] = pd.to_datetime(cambio["data"], dayfirst=True)
cambio["valor"] = cambio["valor"].astype(float)

print(pd.concat([cambio.head(5), cambio.tail(5)]))

          data   valor
0   2023-01-02  5.3430
1   2023-01-03  5.3753
2   2023-01-04  5.4453
3   2023-01-05  5.4020
4   2023-01-06  5.2849
497 2024-12-24  6.1527
498 2024-12-26  6.1650
499 2024-12-27  6.1985
500 2024-12-30  6.1917
501 2024-12-31  6.1917


In [10]:
import pandas as pd
import requests

url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.10813/dados"
params = {
    "formato": "json",
    "dataInicial": "01/01/2023",
    "dataFinal": "31/12/2024"
}

response = requests.get(url, params=params)
df_cambio = pd.DataFrame(response.json())

# Tratamento dos dados do BCB
df_cambio["data"] = pd.to_datetime(df_cambio["data"], dayfirst=True)
df_cambio["usd_brl"] = df_cambio["valor"].astype(float)

# Preencher datas faltantes 
df_cambio = df_cambio.set_index('data')
calendario_completo = pd.date_range("2023-01-01", "2024-12-31", freq="D")
df_cambio = df_cambio.reindex(calendario_completo)
df_cambio['usd_brl'] = df_cambio['usd_brl'].ffill().bfill() 
df_cambio = df_cambio.reset_index().rename(columns={'index': 'data'})

# EXPORTAR PARA CSV
df_cambio.to_csv('historico_cambio_bcb.csv', index=False)
print("Ficheiro 'historico_cambio_bcb.csv' guardado com sucesso!")

Ficheiro 'historico_cambio_bcb.csv' guardado com sucesso!


In [22]:
# Carregamento dos dados
vendas = pd.read_csv('../datasets/vendas_normalizadas.csv')
custos = pd.read_csv('../datasets/custos_importacao.csv')

df_cambio = pd.read_csv('../datasets/historico_cambio_bcb.csv')
df_cambio["data"] = pd.to_datetime(df_cambio["data"])

In [23]:
# Padronizando os dados de vendas e custos
def parsed_date(d):
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            pass
    return pd.NaT

vendas['sale_date'] = vendas['sale_date'].apply(parsed_date)
custos['start_date'] = pd.to_datetime(custos['start_date'])

print("Vendas:")
print(vendas['sale_date'].isnull().sum())
print("Custos:")
print(custos['start_date'].isnull().sum())

Vendas:
0
Custos:
0


In [24]:
# Cruzamento dos dados
# Cruzamento das vendas com o histórico de câmbio
vendas = pd.merge(vendas, df_cambio[['data', 'usd_brl']], left_on='sale_date', right_on='data', how='left')

# Cruzando com o Custo (usando merge_asof para garantir o custo ativo na época)
vendas = vendas.sort_values('sale_date')
custos = custos.sort_values('start_date')

vendas_merged = pd.merge_asof(
    vendas,
    custos[['product_id', 'start_date', 'usd_price', 'product_name']],  # Selecionar apenas as colunas necessárias
    left_on='sale_date',    # Data da venda
    right_on='start_date',  # Data de início do custo
    left_by='id_product',   # ID do produto na venda
    right_by='product_id',  # ID do produto no custo
    direction='backward'    # Garantir que pegamos o custo mais recente antes da venda
)

print(f"Vendas cruzadas com câmbio e custos: {vendas_merged.shape[0]} linhas")
print(vendas_merged.head())

Vendas cruzadas com câmbio e custos: 9895 linhas
     id  id_client  id_product  qtd      total  sale_date       data  usd_brl  \
0  1230         17          91    4  512566.80 2023-01-01 2023-01-01    5.343   
1  2300         30          95    9  596858.40 2023-01-01 2023-01-01    5.343   
2  3131         28         130   13   53873.00 2023-01-01 2023-01-01    5.343   
3  4212          9          96    6  402538.75 2023-01-01 2023-01-01    5.343   
4  4294          7          44    5   51332.30 2023-01-01 2023-01-01    5.343   

   product_id start_date  usd_price  \
0          91 2022-03-16   26303.31   
1          95 2022-11-23   12945.63   
2         130 2021-03-22     749.89   
3          96 2022-09-30   13063.42   
4          44 2022-01-17    1963.02   

                                  product_name  
0  Motor Elétrico Tohatsu Zenith Oceanic 113HP  
1           Motor Diesel Honda Leviathan 133HP  
2   Boia de Arqueamento Delta Zen Boost Vortex  
3      Motor de Popa Tohatsu Boos

> Parte 1 - Calculo e Modelagem
- Calcule o custo total em BRL por transação
- Identifique transações com prejuízo
- Agregue os dados por `id_produto`, gerando:
    1. Receita total (BRL)
    2. Prejuízo total (BRL)
    3. Percentual de perda (`prejuizo_total / receita_total`)

In [28]:
# Cálcular o custo total em BRL por transação
vendas_merged['custo_brl'] = vendas_merged['usd_price'] * vendas_merged['usd_brl']
print(vendas_merged[['id', 'sale_date', 'product_name', 'usd_price', 'usd_brl', 'custo_brl']].head())

     id  sale_date                                 product_name  usd_price  \
0  1230 2023-01-01  Motor Elétrico Tohatsu Zenith Oceanic 113HP   26303.31   
1  2300 2023-01-01           Motor Diesel Honda Leviathan 133HP   12945.63   
2  3131 2023-01-01   Boia de Arqueamento Delta Zen Boost Vortex     749.89   
3  4212 2023-01-01      Motor de Popa Tohatsu Boost Swift 126HP   13063.42   
4  4294 2023-01-01                    GPS Simrad Zen Hydra Peak    1963.02   

   usd_brl     custo_brl  
0    5.343  140538.58533  
1    5.343   69168.50109  
2    5.343    4006.66227  
3    5.343   69797.85306  
4    5.343   10488.41586  


In [31]:
# Transações com prejuízo
vendas_merged['prejuizo'] = vendas_merged['custo_brl'] > vendas_merged['total']
transacoes_prejuizo = vendas_merged[vendas_merged['prejuizo']]
print(transacoes_prejuizo[['id', 'sale_date', 'product_name', 'total', 'custo_brl', 'prejuizo']].head())

      id  sale_date                                 product_name    total  \
31  7226 2023-01-03     Piloto Automático Furuno Core Boost Flux  17283.0   
37  1767 2023-01-03                  Transponder AIS Maré Magnum  33123.0   
43  1832 2023-01-03  Motor Elétrico Torqeedo Velocity Swift 74HP  42989.0   
49  2017 2023-01-04    Motor Elétrico Yamaha Nautic Kraken 133HP  88854.0   
56  4874 2023-01-04               Piloto Automático Furuno Storm  23669.0   

       custo_brl  prejuizo  
31  18345.253864      True  
37  35144.786460      True  
43  43473.222527      True  
49  91186.830441      True  
56  26795.177787      True  


In [34]:
# Agregar os dados por id_product para analisar o desempenho por produto
desempenho_produtos = vendas_merged.groupby('id_product').agg({ 
    'total': 'sum', 
    'custo_brl': 'sum', 
    'prejuizo': 'sum'
}).reset_index()

print(desempenho_produtos.head())

   id_product        total     custo_brl  prejuizo
0           1  17187280.35  2.194111e+06         3
1           2   8862231.15  1.036868e+06         4
2           3   4322136.25  5.890526e+05         4
3           4   1915160.00  2.611329e+05         8
4           5  13494880.35  1.684267e+06         3
